# 고려기프트 발주서 — 이상한 상품명 검출

PDF에서 추출 시 잘려거나 깨진 상품명을 찾습니다.

In [ ]:
import pandas as pd
import re

from config import FLATTENED_ORDERS_PATH

df = pd.read_excel(FLATTENED_ORDERS_PATH)
print(f'전체 행 수: {len(df):,}')
print(f'컬럼: {df.columns.tolist()}')

In [ ]:
# 수량·단가·공급가액·부가세·합계 모두 0이 아닌 행만 필터
# ★ 실제 컬럼명 확인 후 수정
AMOUNT_COLS = ['수량', '단가', '공급가액', '부가세', '합계']
COL_ITEM    = '상품명'
COL_FILE    = '파일명'

# 존재하는 컬럼만 사용
exist_cols = [c for c in AMOUNT_COLS if c in df.columns]
print(f'금액 컬럼 확인: {exist_cols}')

# 금액 컬럼들을 숫자로 변환
for c in exist_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# 모든 금액 컬럼이 0이 아닌 행
mask = (df[exist_cols] != 0).all(axis=1)
valid = df[mask].copy()
print(f'\n유효 행 수: {len(valid):,}개 (전체 {len(df):,}개 중)')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 이상한 상품명 패턴 정의
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
patterns = {
    '쉼표로_시작':        r'^,',                         # ,000mah
    '소수점으로_시작':    r'^\.',                         # .5W
    '괄호만_있음':        r'^[\(\)\[\]]+',               # (20W)
    '숫자로_시작':        r'^\d',                         # 000mah, 5V
    '단위로_시작':        r'^(mah|mAh|AH|W|V|GB|MB|cm|mm|ml|g|kg)\b',  # mAh로 시작
    '너무_짧음':          r'^.{1,3}$',                    # 1~3글자
    '특수문자로_시작':    r'^[^가-힣a-zA-Z0-9\(\[\"\']', # 한글/영문/숫자가 아닌 문자로 시작
    '줄임_의심_영문숫자': r'^[A-Z0-9\(\-\/\.\,\s]{1,10}$',  # 대문자+숫자만 짧게
    '끝이_잘림_의심':     r'[\(\[\-]$',                  # 여는 괄호나 하이픈으로 끝남
}

def detect_issues(name):
    if not isinstance(name, str) or name.strip() == '':
        return ['빈_값']
    s = name.strip()
    found = [label for label, pat in patterns.items() if re.search(pat, s, re.IGNORECASE)]
    return found if found else []

valid['이상_패턴'] = valid[COL_ITEM].apply(detect_issues)
valid['이상여부']  = valid['이상_패턴'].apply(lambda x: len(x) > 0)
valid['이상_패턴_str'] = valid['이상_패턴'].apply(lambda x: ', '.join(x))

bad = valid[valid['이상여부']].copy()
print(f'이상 상품명 행 수: {len(bad):,}개')
print(f'정상 상품명 행 수: {(~valid["이상여부"]).sum():,}개')
print()
print('패턴별 건수:')
for label in patterns:
    cnt = valid['이상_패턴_str'].str.contains(label).sum()
    if cnt:
        print(f'  {label:25s}: {cnt:,}건')

In [ ]:
# 이상 상품명 전체 목록 출력
show_cols = [COL_FILE, COL_ITEM, '이상_패턴_str'] + exist_cols
show_cols = [c for c in show_cols if c in bad.columns]

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)
bad[show_cols].sort_values('이상_패턴_str')

In [ ]:
# 패턴별로 나눠서 보기
for label in patterns:
    subset = bad[bad['이상_패턴_str'].str.contains(label)]
    if len(subset) == 0:
        continue
    print(f'\n{'='*60}')
    print(f'▶ {label}  ({len(subset)}건)')
    print('='*60)
    print(subset[show_cols].head(20).to_string(index=False))

In [ ]:
# 결과 저장
from config import BROKEN_NAMES_PATH

out = BROKEN_NAMES_PATH
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    bad[show_cols].to_excel(writer, sheet_name='이상_전체', index=False)
    for label in patterns:
        subset = bad[bad['이상_패턴_str'].str.contains(label)]
        if len(subset):
            subset[show_cols].to_excel(writer, sheet_name=label[:30], index=False)

print(f'✅ 저장 완료: {out}  ({len(bad)}행)')